# SMILES 与 RDKit：分五步运行

**[▶ 在 Colab 运行](https://colab.research.google.com/github/CCcolab/AI4OrgChem-data-processing/blob/main/skills/organic-smiles-rdkit/organic-smiles-rdkit.ipynb)**

已经在Colab中打开本页？直接按下面步骤操作，无须再点上面的入口。

## 首次运行：账号授权、保存副本与操作步骤

1. **登录并打开**：登录自己的Google账号。GitHub账号须有本私有仓库的访问权限；若Colab提示授权，在“打开笔记本 → GitHub”勾选“包括私有代码库”。核对Colaboratory请求的repo（公私仓库）及gist权限，由本人决定并完成授权；若弹窗没有出现，检查地址栏是否提示拦截。私有仓库不会因此变成公开仓库。
2. **保存副本**：点击顶部“复制到云端硬盘”（或“文件 → 在云端硬盘中保存副本”），在自己的副本中运行和保存。不会修改GitHub原文件。
3. **配置Secret**：点击左侧钥匙“Secret”，为已有的`DEEPSEEK_API_KEY`开启本Notebook的访问权限；首次使用时由本人添加同名Secret。不把密钥值写进代码，不上传.env，不把密钥发到聊天。Colab不能读取你电脑上的.env；本地运行配套模块时，才用`ENV_FILE`选择自己的.env。
4. **依次运行**：点击每个代码单元左侧的 ▶，按①至⑤执行；首次连接选择Python CPU运行时即可。第③步将教学结构、SKILL.md原文和参数发送给DeepSeek，并执行限定RDKit工具，使用你自己的DeepSeek额度。单纯阅读或下载不会调用模型。
5. **修改、保存与下载**：把②中的`RADIUS`从2改为3，确认打印的JSON也是3，再运行③至⑤。每次调用保存新结果。完成后按Ctrl+S或“文件 → 保存”，再用“文件 → 下载 → 下载.ipynb”保留含输出的原生Notebook；结果文件在⑤下载。

| 部分 | 操作 | 运行后核对 |
| --- | --- | --- |
| ① 准备环境 | 点击运行；准备代码默认折叠，可展开 | 首次安装；重复执行检查并复用环境，显示版本和回执 |
| ② 设置参数 | 修改表单，运行参数单元 | 参数JSON与表单一致 |
| ③ 调用Skill | 运行可见的SKILL.md读取与调用代码 | Skill哈希与真实调用回执 |
| ④ 查看结果 | 运行显示单元，按需展开模型解释 | 程序回执、两幅图表及模型原文；文件状态以程序核验和实际文件为准 |
| ⑤ 保存与下载 | 点击需要的下载按钮 | 本轮/全部结果ZIP、SKILL.md、Notebook源码和完整模块ZIP |

完整源码位于配套`runtime`模块，Notebook携带同一经SHA-256核对的模块包，五个代码单元保持简短，不另行要求GitHub Token。附件读取使用Colab内部消息接口；失败会停止并提示。若富文本输出未显示，先保留错误并仅重运行④或⑤，不要反复执行③产生新的模型调用。新RHF不属于本页任务。


In [ ]:
# @title ① 准备环境（首次安装，重复执行复用）
from pathlib import Path
import base64, hashlib, json, sys, tempfile
from google.colab import _message
document = _message.blocking_request("get_ipynb", timeout_sec=20)
if not document or "ipynb" not in document:
    raise RuntimeError("未取得本Notebook附件。请等待页面加载后重试本步骤；不要填写GitHub Token。")
notebook = document["ipynb"]
attachment = notebook["metadata"].get("adp_runtime_bundle")
if not attachment:
    raise RuntimeError("本副本缺少配套模块附件，请重新从本版本运行入口打开。")
payload = base64.b64decode(attachment["zip_base64"], validate=True)
assert hashlib.sha256(payload).hexdigest() == '07a85e2b69ec9f7023a73a13ad61b78610de3240eccec2bf6e6529812dc16038', "配套模块校验失败"
bundle = Path(tempfile.gettempdir()) / ("rdkit-runtime-" + '07a85e2b69ec9f7023a73a13ad61b78610de3240eccec2bf6e6529812dc16038'[:16] + ".zip")
bundle.write_bytes(payload)
sys.path.insert(0, str(bundle))
from colab_workflow import Lab
lab = Lab.prepare(bundle, notebook=notebook)


## ② 参数与输入

样本为14条教学结构；半径默认2，改成3可比较指纹差异。SMARTS查找含OH的羧酸片段；参照005是苯。结构和错误记录全部保留。


In [ ]:
# @title ② 设置分析参数
RADIUS = 2 # @param {type:"integer"}
FP_SIZE = 2048 # @param [1024,2048,4096] {type:"raw"}
INCLUDE_CHIRALITY = True # @param {type:"boolean"}
QUERY_SMARTS = "[CX3](=O)[OX2H1]" # @param {type:"string"}
REFERENCE_ID = "005" # @param {type:"string"}
parameters = dict(radius=RADIUS, fp_size=FP_SIZE, include_chirality=INCLUDE_CHIRALITY,
                  query_smarts=QUERY_SMARTS, reference_id=REFERENCE_ID)
print(json.dumps(parameters, ensure_ascii=False, indent=2))


## ③ 调用 Skill

下面明确读取并传入SKILL.md原文。配套模块负责凭据读取、子进程与超时清理；Agent根据同一Skill调用限定工具。每点击一次会发起一次真实模型与工具任务。


In [ ]:
# @title ③ 读取 SKILL.md 并提交一次真实调用
skill_file = lab.workspace / "SKILL.md"
skill_text = skill_file.read_text(encoding="utf-8")
print("SKILL.md SHA-256:", hashlib.sha256(skill_file.read_bytes()).hexdigest())
run_output = lab.call_skill(skill_text=skill_text, parameters=parameters)


## ④ 结果与解释

查看程序核验回执、分子结构图、Morgan相似度图，展开模型原文。文件状态以程序回执和实际文件为准。


In [ ]:
# @title ④ 查看程序回执、模型解释和图表
lab.show_results(run_output)


## ⑤ 下载与保存

下载本轮/全部结果ZIP、SKILL.md、Notebook源码及完整模块包。保留含实际运行输出的Notebook，请使用“文件→保存”，再用“文件→下载→下载.ipynb”。


In [ ]:
# @title ⑤ 保存与下载
lab.show_downloads(run_output)
